In [1]:
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_DIR = "models"
ARTIFACT_DIR = "artifacts"
PROCESSED_DIR = "datasets/processed"
OUT_DIR = "docs/Week07"

import os
os.makedirs(OUT_DIR, exist_ok=True)

gpu_devices = tf.config.list_physical_devices('GPU')
print("TensorFlow chạy trên GPU:", len(gpu_devices) > 0, gpu_devices)

TensorFlow chạy trên GPU: False []


In [33]:
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow import keras

rf_model = joblib.load(f"{MODEL_DIR}/tier1_rf.pkl")
svm_model = joblib.load(f"{MODEL_DIR}/tier1_svm.pkl")

class AdditiveAttention(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.supports_masking = True

    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(feat_dim, self.units),
                                  initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="b", shape=(self.units,),
                                  initializer="zeros", trainable=True)
        self.v = self.add_weight(name="v", shape=(self.units, 1),
                                  initializer="glorot_uniform", trainable=True)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        score = tf.tanh(tf.tensordot(inputs, self.W, axes=[[2], [0]]) + self.b)
        score = tf.tensordot(score, self.v, axes=[[2], [0]])
        score = tf.squeeze(score, axis=-1)
        if mask is not None:
            mask = tf.cast(mask, dtype=score.dtype)
            score = score + (1.0 - mask) * -1e9
        alpha = tf.nn.softmax(score, axis=1)
        alpha = tf.expand_dims(alpha, axis=-1)
        context = tf.reduce_sum(inputs * alpha, axis=1)
        return context, alpha

    def compute_mask(self, inputs, mask=None):
        return None

VOCAB_SIZE = 96
MAX_LEN = 250
EMBED_DIM = 64
LSTM_UNITS = 64
ATTN_UNITS = 64

def build_tier3_model():
    inputs = Input(shape=(MAX_LEN,), name="input_sequence")
    x = layers.Embedding(
        input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
        mask_zero=True, name="embedding"
    )(inputs)
    lstm_out = layers.LSTM(LSTM_UNITS, return_sequences=True, name="lstm")(x)
    context, alpha = AdditiveAttention(units=ATTN_UNITS, name="additive_attention")(lstm_out)
    dropped = layers.Dropout(0.2, name="dropout")(context)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(dropped)
    return Model(inputs=inputs, outputs=outputs, name="tier3_lstm_attention")

tier2_model = keras.models.load_model(f"{MODEL_DIR}/tier2_lstm.h5")

tier3_model = build_tier3_model()

# Chặn cứng trước khi load_weights: nếu kiến trúc rebuild sai một chi tiết nào đó
# (thứ tự layer, thiếu Dropout, sai units...), số tham số sẽ lệch ngay tại đây.
assert tier3_model.count_params() == 43457, (
    f"Số tham số rebuild = {tier3_model.count_params()}, không khớp 43.457 đã ghi nhận."
)

tier3_model.load_weights(f"{MODEL_DIR}/tier3_lstm_attention.h5")

In [38]:
import numpy as np
X_test = np.load("../datasets/processed/test_sequences.npy")

print("X_test shape:", X_test.shape)

X_test shape: (6181, 250)


In [39]:
sample_probs = tier3_model.predict(X_test[:10], verbose=0)
print(sample_probs.ravel())

[2.40313215e-03 1.57003626e-04 1.60035619e-03 1.36974305e-02
 1.36631436e-03 1.19196338e-04 2.75404588e-03 1.25094652e-04
 9.99824584e-01 8.09302926e-03]


In [ ]:
import json

with open(f"{ARTIFACT_DIR}/tier3_training_history.json") as f:
    tier3_hist = json.load(f)

print(tier3_hist.keys())
# Nếu có key kiểu 'epoch_time' hoặc tương tự -> in luôn để lấy tổng thời gian thật
# Nếu chỉ có loss/val_loss/accuracy... -> in độ dài để biết SỐ EPOCH THỰC TẾ đã chạy
for k, v in tier3_hist.items():
    print(k, len(v) if isinstance(v, list) else v)